# IEEE Minesweepers 2026 — Retrain YOLOv8s
Retrain with more angles to improve detection from all directions.

**Upload to Google Drive:**
1. `best.pt` (from models folder)
2. `ms_detect` folder (train/valid/test)
3. `new_images` folder (your new photos, no labels needed)

In [ ]:
# Step 1: Install
!pip install ultralytics albumentations

In [ ]:
# Step 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 3: Copy files from Drive and merge into one dataset
# Files are in My Drive > Minesweeper

!cp /content/drive/MyDrive/Minesweeper/best.pt /content/best.pt
!cp -r /content/drive/MyDrive/Minesweeper/ms_detect /content/ms_detect
!mkdir -p /content/new_images
!cp -r /content/drive/MyDrive/Minesweeper/new_images/* /content/new_images/

# Create unified dataset folder
!mkdir -p /content/dataset/train/images
!mkdir -p /content/dataset/train/labels
!mkdir -p /content/dataset/valid/images
!mkdir -p /content/dataset/valid/labels
!mkdir -p /content/dataset/test/images
!mkdir -p /content/dataset/test/labels

# Copy original train/valid/test
!cp /content/ms_detect/ms_detect/train/images/* /content/dataset/train/images/
!cp /content/ms_detect/ms_detect/train/labels/* /content/dataset/train/labels/
!cp /content/ms_detect/ms_detect/valid/images/* /content/dataset/valid/images/
!cp /content/ms_detect/ms_detect/valid/labels/* /content/dataset/valid/labels/
!cp /content/ms_detect/ms_detect/test/images/* /content/dataset/test/images/
!cp /content/ms_detect/ms_detect/test/labels/* /content/dataset/test/labels/

import os
print(f'Original train: {len(os.listdir("/content/dataset/train/images"))} images')
print(f'Valid: {len(os.listdir("/content/dataset/valid/images"))} images')

new_count = len([f for f in os.listdir("/content/new_images") if f.lower().endswith((".jpg",".jpeg",".png"))])
print(f'New images to add: {new_count}')

In [ ]:
# Step 4: Auto-label new images using current model
from ultralytics import YOLO
import os
import cv2
import shutil

model = YOLO('/content/best.pt')

new_img_dir = '/content/new_images'
out_img_dir = '/content/dataset/train/images'
out_lbl_dir = '/content/dataset/train/labels'

labeled = 0
skipped = 0

for fname in os.listdir(new_img_dir):
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    
    img_path = os.path.join(new_img_dir, fname)
    results = model(img_path, conf=0.3, verbose=False)
    
    boxes = results[0].boxes
    if len(boxes) == 0:
        skipped += 1
        print(f'  SKIPPED (no detection): {fname}')
        continue
    
    new_fname = 'new_' + fname
    shutil.copy(img_path, os.path.join(out_img_dir, new_fname))
    
    label_path = os.path.join(out_lbl_dir, os.path.splitext(new_fname)[0] + '.txt')
    with open(label_path, 'w') as f:
        for box in boxes:
            xywhn = box.xywhn[0].tolist()
            f.write(f'0 {xywhn[0]:.6f} {xywhn[1]:.6f} {xywhn[2]:.6f} {xywhn[3]:.6f}\n')
    
    labeled += 1
    print(f'  OK: {fname} -> {len(boxes)} detection(s)')

print(f'\n=== Auto-labeling done: {labeled} labeled, {skipped} skipped ===')
print(f'Total train images now: {len(os.listdir(out_img_dir))}')

In [ ]:
# Step 5: Review auto-labels (spot check)
import matplotlib.pyplot as plt
import matplotlib.patches as patches

out_img_dir = '/content/dataset/train/images'
out_lbl_dir = '/content/dataset/train/labels'

check_files = [f for f in os.listdir(out_img_dir) if f.startswith('new_')][:5]

fig, axes = plt.subplots(1, min(5, len(check_files)), figsize=(20, 4))
if len(check_files) == 1:
    axes = [axes]

for ax, fname in zip(axes, check_files):
    img = cv2.imread(os.path.join(out_img_dir, fname))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)
    
    label_path = os.path.join(out_lbl_dir, os.path.splitext(fname)[0] + '.txt')
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                cx, cy, bw, bh = float(parts[1])*w, float(parts[2])*h, float(parts[3])*w, float(parts[4])*h
                rect = patches.Rectangle((cx-bw/2, cy-bh/2), bw, bh, linewidth=2, edgecolor='r', facecolor='none')
                ax.add_patch(rect)
    ax.set_title(fname, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()
print('Check that boxes are correct.')

In [ ]:
# Step 5.5: Color Augmentation on ALL training images (original + new)
import cv2
import numpy as np
import os
import shutil

img_dir = '/content/dataset/train/images'
lbl_dir = '/content/dataset/train/labels'

color_shifts = {
    'rusty':   (15, 60, 30),
    'blue':    (-60, 40, 20),
    'green':   (40, 50, 10),
    'dark':    (0, 0, -50),
    'bright':  (0, 0, 50),
    'desat':   (0, -80, 0),
}

all_images = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png')) and not f.startswith('aug_')]
created = 0

for fname in all_images:
    img_path = os.path.join(img_dir, fname)
    lbl_path = os.path.join(lbl_dir, os.path.splitext(fname)[0] + '.txt')
    
    if not os.path.exists(lbl_path):
        continue
    
    img = cv2.imread(img_path)
    if img is None:
        continue
    
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    
    for shift_name, (h_shift, s_shift, v_shift) in color_shifts.items():
        new_hsv = hsv.copy()
        new_hsv[:, :, 0] = (new_hsv[:, :, 0] + h_shift) % 180
        new_hsv[:, :, 1] = np.clip(new_hsv[:, :, 1] + s_shift, 0, 255)
        new_hsv[:, :, 2] = np.clip(new_hsv[:, :, 2] + v_shift, 0, 255)
        
        new_img = cv2.cvtColor(new_hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
        
        new_fname = f'aug_{shift_name}_{fname}'
        cv2.imwrite(os.path.join(img_dir, new_fname), new_img)
        
        new_lbl = os.path.splitext(new_fname)[0] + '.txt'
        shutil.copy(lbl_path, os.path.join(lbl_dir, new_lbl))
        created += 1

total = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
print(f'Color augmentation done: {created} new images created')
print(f'Total training images now: {total}')

In [ ]:
# Step 5.7: Preview color augmentation — one sample from each type
import matplotlib.pyplot as plt
import cv2
import os

img_dir = '/content/dataset/train/images'
shift_names = ['rusty', 'blue', 'green', 'dark', 'bright', 'desat']

# Find one original image
original = [f for f in os.listdir(img_dir) if not f.startswith('aug_') and not f.startswith('new_') and f.lower().endswith(('.jpg','.jpeg','.png'))][0]

# Find one new image (if exists)
new_imgs = [f for f in os.listdir(img_dir) if f.startswith('new_') and f.lower().endswith(('.jpg','.jpeg','.png'))]
new_sample = new_imgs[0] if new_imgs else None

rows = 2 if new_sample else 1
fig, axes = plt.subplots(rows, 7, figsize=(24, 4 * rows))
if rows == 1:
    axes = [axes]

# Row 1: Original training image + its augmented versions
titles = ['Original (train)', 'Rusty', 'Blue', 'Green', 'Dark', 'Bright', 'Desaturated']
files_row1 = [original] + [f'aug_{name}_{original}' for name in shift_names]

for ax, fname, title in zip(axes[0], files_row1, titles):
    path = os.path.join(img_dir, fname)
    if os.path.exists(path):
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, 'Not found', ha='center', va='center')
    ax.set_title(title, fontsize=10)
    ax.axis('off')

# Row 2: New image + its augmented versions
if new_sample:
    titles2 = ['Original (new)', 'Rusty', 'Blue', 'Green', 'Dark', 'Bright', 'Desaturated']
    files_row2 = [new_sample] + [f'aug_{name}_{new_sample}' for name in shift_names]

    for ax, fname, title in zip(axes[1], files_row2, titles2):
        path = os.path.join(img_dir, fname)
        if os.path.exists(path):
            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, 'Not found', ha='center', va='center')
        ax.set_title(title, fontsize=10)
        ax.axis('off')

plt.suptitle('Color Augmentation Preview', fontsize=14)
plt.tight_layout()
plt.show()

total = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
for name in shift_names:
    count = len([f for f in os.listdir(img_dir) if f.startswith(f'aug_{name}_')])
    print(f'  {name}: {count} images')
print(f'\nTotal training images: {total}')

In [ ]:
# Step 6: Fix data.yaml paths
import yaml
import os

data_config = {
    'names': ['mine'],
    'nc': 1,
    'train': '/content/dataset/train/images',
    'val': '/content/dataset/valid/images',
    'test': '/content/dataset/test/images'
}

with open('/content/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

train_count = len([f for f in os.listdir('/content/dataset/train/images') if f.lower().endswith(('.jpg','.jpeg','.png'))])
val_count = len([f for f in os.listdir('/content/dataset/valid/images') if f.lower().endswith(('.jpg','.jpeg','.png'))])
print(f'Train: {train_count} images')
print(f'Valid: {val_count} images')

In [ ]:
# Step 7: Retrain
from ultralytics import YOLO

model = YOLO('/content/best.pt')

results = model.train(
    data='/content/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=20,
    augment=True,
    flipud=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    scale=0.5,
    name='minesweeper_v2'
)

In [ ]:
# Step 8: Check training results
from IPython.display import Image, display

display(Image(filename='runs/detect/minesweeper_v2/results.png'))
print('\nConfusion Matrix:')
display(Image(filename='runs/detect/minesweeper_v2/confusion_matrix.png'))

In [ ]:
# Step 8.5: Test on test images — run detection and show results
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2
import os

best_model = YOLO('runs/detect/minesweeper_v2/weights/best.pt')

test_dir = '/content/dataset/test/images'
test_images = [f for f in os.listdir(test_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]

# Run validation on test set
print("=== Test Set Evaluation ===")
metrics = best_model.val(data='/content/data.yaml', split='test')
print(f"\nmAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

# Show predictions on test images
cols = min(5, len(test_images))
rows = min(4, (len(test_images) + cols - 1) // cols)
show_count = rows * cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
if rows == 1:
    axes = [axes]
if cols == 1:
    axes = [[ax] for ax in axes]

for idx in range(show_count):
    r, c = idx // cols, idx % cols
    if idx >= len(test_images):
        axes[r][c].axis('off')
        continue
    
    img_path = os.path.join(test_dir, test_images[idx])
    results = best_model(img_path, conf=0.3, verbose=False)
    
    img = results[0].plot()
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    axes[r][c].imshow(img)
    
    n_det = len(results[0].boxes)
    confs = [f"{b.conf[0]:.0%}" for b in results[0].boxes]
    axes[r][c].set_title(f'{test_images[idx]}\n{n_det} det: {", ".join(confs)}', fontsize=8)
    axes[r][c].axis('off')

plt.suptitle('Test Set Predictions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Step 9: Export to TFLite INT8 for Pi 4
best_model = YOLO('runs/detect/minesweeper_v2/weights/best.pt')
best_model.export(format='tflite', int8=True, imgsz=320)

In [ ]:
# Step 10: Download
from google.colab import files

files.download('runs/detect/minesweeper_v2/weights/best.pt')

import glob
tflite_files = glob.glob('runs/detect/minesweeper_v2/weights/*int8.tflite')
if tflite_files:
    files.download(tflite_files[0])